In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
import numpy as np
import pyccl as ccl
import yaml
import sacc
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"

# This notebook creates the sacc file containing n(z), w(theta) and covariance.

first_run = True # compute w(theta) under the Limber approximation. Do not store the covariance since it does not exist yet
second_run = not first_run # compute w(theta) without using Limber. Store the covariance created after running 'run_cov_wtheta.sh'

if first_run:
    l_limber = -1
else:
    # l_limber = "auto" # non-Limber (much slower)
    l_limber = -1 
    # Since this repo's goal is to show how to compute a covariance with TJPCov, we will not use non-Limber for the w(theta). When computing the covariance, 
    # it doesn't matter what w(theta) datavector is stored in the SACC file (Limber, non-Limber, from TreeCorr, etc.), TJPCov takes the binning in theta 
    # and uses the same one, but it doesn't use the w(theta) itself

# ============================================================
# 1. Load n(z)
# ============================================================

nz_file = f"nz_files/nz_DNFpdf_shift_stretch_wrtclusteringz1-4_wrtVIPERS5-6_v2.txt"

data = np.loadtxt(nz_file)

z = data[:, 0]
nz = np.array([data[:, i+1] for i in range(len(data.T)-1)]).T

print("z shape:", z.shape)
print("nz shape:", nz.shape)

# Five tomographic bins
nbins = len(nz.T)

# Normalize each n(z)
for i in range(nbins):
    norm = np.trapz(nz[:, i], z)
    nz[:, i] /= norm

# Galaxy bias for each bin
bias = np.ones(nbins)

# ============================================================
# 2. Cosmology
# ============================================================

with open("cosmo_planck.yaml", "r") as f:
    cosmo_params = yaml.safe_load(f)

cosmo = ccl.Cosmology(**cosmo_params)

# ============================================================
# 3. Create the five galaxy tracers
# ============================================================

tracers = []

for i in range(nbins):

    tracer = ccl.NumberCountsTracer(
        cosmo,
        has_rsd=True,
        dndz=(z, nz[:, i]),
        bias=(z, bias[i] * np.ones_like(z)),
    )

    tracers.append(tracer)

# ============================================================
# 4. Multipoles
# ============================================================

ell = np.arange(2, 20000)

# ============================================================
# 5. Calculate C_ell for autocorrelations only
# ============================================================

cls = {}

for i in range(nbins):

    print(f"Computing C_ell for bin {i+1} x {i+1}")

    cl = ccl.angular_cl(
        cosmo,
        tracers[i],
        tracers[i],
        ell,
        # limber_integration_method='spline',
        l_limber=l_limber,
    )

    cls[i] = cl

# ============================================================
# 6. Calculate w(theta)
# ============================================================

theta = np.array([0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5, 1.7, 1.9, 2.1, 2.3, 2.5, 2.7,
                  2.9, 3.1, 3.3, 3.5, 3.7, 3.9, 4.1, 4.3, 4.5, 4.7, 4.9]) # same as in DES Y6 BAO

wtheta = {}

for i, cl in cls.items():

    print(f"Computing w(theta) for bin {i+1} x {i+1}")

    w = ccl.correlations.correlation(
        cosmo,
        ell=ell,
        C_ell=cl,
        theta=theta,
        type="NN",
        method='fftlog',
    )

    wtheta[i] = w

# ============================================================
# 7. Plot: 5 subplots
# ============================================================

fig, axes = plt.subplots(
    nbins, 1,
    figsize=(7, 3 * nbins),
    sharex=True,
)

for i, ax in enumerate(axes):

    ax.plot(
        theta,
        100 * theta * wtheta[i],
    )

    ax.set_ylabel(r"$w_{" + f"{i+1}{i+1}" + r"}(\theta)$")
    ax.set_title(f"Tomographic bin {i+1}")

axes[-1].set_xlabel(r"$\theta$ [deg]")

plt.tight_layout()

# ============================================================
# 8. Save w(theta) to SACC
# ============================================================

# Create SACC tracers from the same n(z) used for the calculation
sacc_tracers = []

for i in range(nbins):
    tracer = sacc.tracers.NZTracer(
        f"lens{i}",
        z,
        nz[:, i],
        quantity="galaxy_density",
    )
    sacc_tracers.append(tracer)

# Create SACC object
s = sacc.Sacc()

# Add tracers
for tracer in sacc_tracers:
    s.add_tracer_object(tracer)

# Add autocorrelation w(theta) for each tomographic bin
for i in range(nbins):
    for theta_i, w_i in zip(theta, wtheta[i]):
        s.add_data_point(
            "galaxy_density_xi",
            [sacc_tracers[i].name, sacc_tracers[i].name],
            w_i,
            theta=theta_i * 60,
        )

if second_run:
    # Add covariance. This has to be done during the second run, when the covariance has already
    # been calculated using the wtheta sacc file that does not include the covariance.
    cov = np.loadtxt(f"cov_tjpcov/cov_wtheta_DESY6BAO.txt")
    s.add_covariance(cov)
    print("Covariance added for the second run!")

# Save
s.save_fits(
    f"wtheta_sacc/wtheta_DESY6BAO.fits",
    overwrite=True,
)
